# 05 — Beat Labeling

Defining and verifying the label scheme for our segmented heartbeats, using MIT-BIH's annotation symbols. No feature extraction or ML yet — this notebook is purely about getting the label definition right and documented.

## Inspecting real annotation symbols first

Before choosing a scheme, look at what's actually in the data — across more than one record, since a single record badly understates how rare some symbols are.

In [1]:
import wfdb
import collections

INSPECTION_RECORDS = ["100", "101", "103", "108", "109", "200", "217"]

total_counts = collections.Counter()
for rec_name in INSPECTION_RECORDS:
    ann = wfdb.rdann(rec_name, "atr", pn_dir="mitdb")
    total_counts.update(ann.symbol)

print("Aggregated annotation symbol counts across", INSPECTION_RECORDS, ":")
for symbol, count in total_counts.most_common():
    print(f"  '{symbol}': {count}")

Aggregated annotation symbol counts across ['100', '101', '103', '108', '109', '200', '217'] :
  'N': 9907
  'L': 2492
  '/': 1542
  'V': 1044
  'f': 260
  '+': 220
  '~': 100
  'A': 72
  '|': 13
  'x': 11
  'F': 6
  'Q': 2
  'j': 1


## Choosing the classification problem: Normal vs. Abnormal

Two options were considered:

- **Normal vs. Abnormal (binary):** merging into two buckets gives roughly
  12,400 Normal vs 2,926 Abnormal across these 7 records (~19% minority
  class), imbalanced but workable with basic imbalance-aware ML.
- **Small multi-class (e.g. Normal / Supraventricular-ectopic /
  Ventricular-ectopic):** the real counts show V (1044) vs. A (72) is
  already a ~14:1 imbalance *between the two minority classes*, before even
  comparing to Normal — and record-level splitting means a test set could
  easily end up with very few A-class examples by chance.

**Decision: binary Normal vs. Abnormal**, given the project's timeline.
Multi-class is documented as a Future Improvement rather than attempted now
— a scope decision, not a shortcut taken silently.

## The mapping — the AAMI EC57 standard grouping

This symbol-to-superclass grouping is the field standard used across ECG
beat-classification literature (e.g. de Chazal et al.), just merged down to
two buckets for this project:

- **Normal** (AAMI N-class — beats of normal sinus origin, including
  conduction-pathway variants that still originate normally): `N`, `L`, `R`,
  `e`, `j`
- **Abnormal** (AAMI S + V + F classes — ectopic beats, i.e. beats
  originating from an abnormal focus): `A`, `a`, `J`, `S`, `V`, `E`, `F`
- **Excluded — paced/unclassifiable** (AAMI Q-class): `/`, `f`, `Q`. Paced
  beats are a device-driven artifact (a pacemaker spike), not a
  physiological arrhythmia — classifying them is a genuinely different
  problem, so they're out of scope here rather than folded into "Abnormal".
- **Excluded — not a beat at all**: `+` (rhythm change marker), `~`
  (signal quality marker), `|` (isolated artifact), `x` (non-conducted
  P-wave — a P-wave event with no QRS/R-peak to anchor a beat window on).

In [2]:
import sys
sys.path.append("..")

from src.feature_extraction import (ABNORMAL_SYMBOLS, NON_BEAT_SYMBOLS, NORMAL_SYMBOLS, PACED_SYMBOLS,
                                    map_symbol_to_label)

# The mapping decided in this notebook lives in src/feature_extraction.py so the pipeline and the notebooks share it.
for name, group in [("Normal", NORMAL_SYMBOLS), ("Abnormal", ABNORMAL_SYMBOLS),
                    ("Excluded (paced/unclassifiable)", PACED_SYMBOLS), ("Excluded (not a beat)", NON_BEAT_SYMBOLS)]:
    print(f"{name:<32}", sorted(group))

## Verifying the mapping against real data

Apply it to every annotation in the inspection records and check nothing
falls through as "unrecognized", an unrecognized symbol should be
surfaced explicitly, never silently guessed at.

In [3]:
label_counts = collections.Counter()
unrecognized = collections.Counter()

for rec_name in INSPECTION_RECORDS:
    ann = wfdb.rdann(rec_name, "atr", pn_dir="mitdb")
    for sym in ann.symbol:
        label = map_symbol_to_label(sym)
        label_counts[label] += 1
        if label == "Excluded (unrecognized symbol)":
            unrecognized[sym] += 1

print("Resulting label distribution:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")

if unrecognized:
    print("\nUnrecognized symbols (need explicit handling before proceeding):", dict(unrecognized))
else:
    print("\nNo unrecognized symbols — every annotation in this sample is accounted for.")

Resulting label distribution:
  Normal: 12400
  Excluded (paced/unclassifiable): 1804
  Abnormal: 1122
  Excluded (not a beat): 344

No unrecognized symbols — every annotation in this sample is accounted for.


## Handling beats that can't be reliably assigned

Three distinct situations, all handled the same way:

1. **Non-beat annotations** (`+`, `~`, `|`, `x`, etc.) — these were never
   heartbeats to begin with, so they're dropped before labeling even starts.
2. **Explicitly out-of-scope beat types** (paced/unclassifiable) — real
   beats, but a different problem than the one we've scoped; excluded by
   deliberate decision, documented here and in the eventual README.
3. **A detected R-peak with no matching annotation nearby** — this will
   happen for false-positive detections (like the noise-triggered spurious
   peak we found in `03_rpeak_detection.ipynb`). Forcing a label onto a beat
   we have no ground truth for would inject label noise into training data.
   These get dropped when we assign labels to the segmented beats next.
4. **Any symbol not covered by this table at all** — the function above
   returns an explicit `"Excluded (unrecognized symbol)"` rather than
   defaulting to a guess, precisely so a genuinely new symbol shows up as a
   visible count to investigate, not a silent misclassification.

**Next step (not yet done here):** apply this mapping to the actual
segmented beats from `04_heartbeat_segmentation.ipynb` — for each
`beat_r_peak_samples` entry, find the nearest annotation within a small
tolerance and assign its label, dropping any beat that doesn't get a clean
match. That, plus selecting a small pool of records with enough Abnormal
beats in each, is what Stage 8 (feature extraction) will build on.